# Instagram Followers Data Bot 🤖

This notebook automates the extraction of follower data from any public Instagram account.

## Features:
- 🔐 Secure login with 2FA support
- 📊 Extract complete followers list from any target account
- 📈 Collect detailed profile data (posts, followers, following) for each follower
- 💾 Export data to CSV format

## Setup:
1. Copy `.env.example` to `.env`
2. Fill in your Instagram credentials
3. Set your target account name
4. Run all cells sequentially

## 1. Import Dependencies

In [ ]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# Load environment variables
load_dotenv()

print("✅ All dependencies imported successfully!")

## 2. Configuration

Credentials are loaded from `.env` file for security.

In [ ]:
# Load credentials from environment variables
INSTAGRAM_URL = 'https://www.instagram.com'
USERNAME = os.getenv('INSTAGRAM_USERNAME', '')
PASSWORD = os.getenv('INSTAGRAM_PASSWORD', '')
TARGET_ACCOUNT = os.getenv('TARGET_ACCOUNT', '')

# Validate credentials are set
if not USERNAME or not PASSWORD:
    raise ValueError("❌ Please set INSTAGRAM_USERNAME and INSTAGRAM_PASSWORD in your .env file")
if not TARGET_ACCOUNT:
    raise ValueError("❌ Please set TARGET_ACCOUNT in your .env file")

print(f"✅ Configuration loaded for target account: @{TARGET_ACCOUNT}")

## 3. Helper Functions

In [ ]:
def wait_and_find(driver, by, value, timeout=10):
    """Wait for element to be present and return it."""
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((by, value))
        )
        return element
    except TimeoutException:
        return None

def safe_click(driver, by, value, timeout=5):
    """Safely click an element if it exists."""
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((by, value))
        )
        element.click()
        return True
    except (TimeoutException, NoSuchElementException):
        return False

def check_element_exists(driver, xpath):
    """Check if an element exists on the page."""
    try:
        driver.find_element(By.XPATH, xpath)
        return True
    except NoSuchElementException:
        return False

print("✅ Helper functions defined!")

## 4. Initialize WebDriver & Login

In [ ]:
# Initialize Chrome WebDriver
driver = webdriver.Chrome()
driver.get(INSTAGRAM_URL)
driver.maximize_window()
print("🌐 Browser opened, waiting for Instagram to load...")

# Wait for page load and enter username
username_field = wait_and_find(driver, By.XPATH, "//input[@aria-label='Phone number, username, or email']", timeout=15)
if username_field:
    username_field.send_keys(USERNAME)
    print("✅ Username entered")

# Enter password
password_field = wait_and_find(driver, By.XPATH, "//input[@aria-label='Password']", timeout=10)
if password_field:
    password_field.send_keys(PASSWORD)
    print("✅ Password entered")

# Click login button
time.sleep(2)
if safe_click(driver, By.XPATH, "//button[@type='submit']"):
    print("✅ Login button clicked")

# Handle 2FA if prompted
time.sleep(5)
try:
    auth_field = wait_and_find(driver, By.XPATH, "//input[@aria-label='Verification code']", timeout=5)
    if auth_field:
        auth_code = input("🔐 Enter your 2FA code: ")
        auth_field.send_keys(auth_code)
        safe_click(driver, By.XPATH, "//button[contains(text(), 'Confirm')]")
        print("✅ 2FA code submitted")
except:
    pass

# Handle "Save Login Info" prompts
time.sleep(5)
safe_click(driver, By.XPATH, "//button[text()='Not Now']")
time.sleep(3)
safe_click(driver, By.XPATH, "//button[text()='Not Now']")

print("✅ Login completed!")

## 5. Navigate to Target Account

In [ ]:
# Navigate directly to target profile
profile_url = f"{INSTAGRAM_URL}/{TARGET_ACCOUNT}/"
driver.get(profile_url)
time.sleep(5)
print(f"🔍 Navigated to @{TARGET_ACCOUNT}'s profile")

# Get profile information
try:
    posts_element = wait_and_find(driver, By.XPATH, "//ul/li[1]//span", timeout=10)
    followers_element = wait_and_find(driver, By.XPATH, "//ul/li[2]//span", timeout=10)
    following_element = wait_and_find(driver, By.XPATH, "//ul/li[3]//span", timeout=10)
    
    posts_count = posts_element.text if posts_element else "N/A"
    followers_count = followers_element.text if followers_element else "N/A"
    following_count = following_element.text if following_element else "N/A"
    
    print(f"📊 Profile Stats for @{TARGET_ACCOUNT}:")
    print(f"   Posts: {posts_count}")
    print(f"   Followers: {followers_count}")
    print(f"   Following: {following_count}")
except Exception as e:
    print(f"⚠️ Could not retrieve profile stats: {e}")

## 6. Extract Followers List

In [ ]:
# Click on followers to open the modal
if followers_element:
    followers_element.click()
    time.sleep(3)
    print("📂 Followers modal opened")

# Get total followers count
try:
    followers_count_num = int(followers_count.replace(',', ''))
    print(f"📈 Total followers to scrape: {followers_count_num}")
except:
    followers_count_num = None
    print("⚠️ Could not parse followers count, will scroll until no new followers")

# Scroll to load all followers
followers_list = []
last_count = 0
no_change_count = 0

while True:
    # Find all follower elements
    follower_elements = driver.find_elements(By.XPATH, "//a[contains(@href, '/') and not(contains(@href, 'explore'))]")[2:]  # Skip nav links
    
    # Alternative: Try to find followers in the modal
    try:
        scrollable_div = wait_and_find(driver, By.XPATH, "//div[@role='dialog']//div[contains(@style, 'overflow') or contains(@class, 'scroll')]", timeout=5)
        if scrollable_div:
            scrollable_div.send_keys(Keys.END)
    except:
        pass
    
    # Send PAGE_DOWN to load more
    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
    time.sleep(1)
    
    # Extract follower usernames
    current_followers = [elem.text for elem in follower_elements if elem.text and elem.text not in ['', ' ']]
    
    if len(current_followers) == last_count:
        no_change_count += 1
        if no_change_count >= 5:  # Stop if no change after 5 attempts
            print("✅ All followers loaded!")
            break
    else:
        no_change_count = 0
    
    last_count = len(current_followers)
    
    # Progress update
    if followers_count_num and len(current_followers) >= followers_count_num:
        print(f"✅ Reached target: {len(current_followers)} followers")
        break
    
    print(f"⏳ Loading followers... ({len(current_followers)} loaded)")

# Remove duplicates and store final list
full_list_of_followers = list(dict.fromkeys(current_followers))
print(f"\n✅ Total unique followers extracted: {len(full_list_of_followers)}")

## 7. Verify Extracted Data

In [ ]:
# Display sample of extracted followers
print(f"📋 Total followers extracted: {len(full_list_of_followers)}")
print(f"\nSample followers (first 10):")
for i, follower in enumerate(full_list_of_followers[:10], 1):
    print(f"   {i}. @{follower}")

## 8. Save Followers List to CSV

In [ ]:
# Save followers list to CSV
output_filename = f"{TARGET_ACCOUNT}_followers_list.csv"
df = pd.DataFrame({"Username": full_list_of_followers})
df.to_csv(output_filename, index=False)
print(f"💾 Followers list saved to: {output_filename}")

## 9. Collect Detailed Follower Data

This section visits each follower's profile to collect their stats (posts, followers, following).

⚠️ **Warning:** This process can take a long time for accounts with many followers.

In [ ]:
# Initialize data structure for detailed follower info
follower_details = {
    "Username": [],
    "Posts": [],
    "Followers": [],
    "Following": [],
    "Profile_Type": []
}

# Load the followers list
df = pd.read_csv(output_filename)
total_followers = len(df['Username'])

print(f"🔍 Starting detailed data collection for {total_followers} followers...")
print("⚠️ This may take a while. You can interrupt at any time.\n")

for index, username in enumerate(df['Username'], 1):
    try:
        # Navigate to follower's profile
        profile_url = f"{INSTAGRAM_URL}/{username}/"
        driver.get(profile_url)
        time.sleep(3)  # Be respectful to Instagram's servers
        
        # Check if page exists
        if check_element_exists(driver, "//h2[contains(text(), 'Sorry')]"):
            # Page not available
            follower_details["Username"].append(username)
            follower_details["Posts"].append("N/A")
            follower_details["Followers"].append("N/A")
            follower_details["Following"].append("N/A")
            follower_details["Profile_Type"].append("Unavailable")
            continue
        
        # Check if profile is private
        is_private = check_element_exists(driver, "//h2[contains(text(), 'Follow')]")
        profile_type = "Private" if is_private else "Public"
        
        # Extract profile stats
        try:
            posts = wait_and_find(driver, By.XPATH, "//ul/li[1]//span", timeout=5)
            followers = wait_and_find(driver, By.XPATH, "//ul/li[2]//span", timeout=5)
            following = wait_and_find(driver, By.XPATH, "//ul/li[3]//span", timeout=5)
            
            follower_details["Username"].append(username)
            follower_details["Posts"].append(posts.text if posts else "N/A")
            follower_details["Followers"].append(followers.text if followers else "N/A")
            follower_details["Following"].append(following.text if following else "N/A")
            follower_details["Profile_Type"].append(profile_type)
            
        except Exception as e:
            follower_details["Username"].append(username)
            follower_details["Posts"].append("Error")
            follower_details["Followers"].append("Error")
            follower_details["Following"].append("Error")
            follower_details["Profile_Type"].append("Error")
        
        # Progress update every 10 profiles
        if index % 10 == 0:
            print(f"⏳ Progress: {index}/{total_followers} profiles processed ({(index/total_followers)*100:.1f}%)")
            
    except Exception as e:
        print(f"❌ Error processing @{username}: {e}")
        continue

print(f"\n✅ Data collection completed! Processed {len(follower_details['Username'])} profiles.")

## 10. Save Final Detailed Data

In [ ]:
# Create DataFrame and save to CSV
df_final = pd.DataFrame(follower_details)
final_output = f"{TARGET_ACCOUNT}_followers_detailed.csv"
df_final.to_csv(final_output, index=False)

print(f"💾 Detailed data saved to: {final_output}")
print(f"\n📊 Summary:")
print(f"   Total profiles: {len(df_final)}")
print(f"   Public profiles: {len(df_final[df_final['Profile_Type'] == 'Public'])}")
print(f"   Private profiles: {len(df_final[df_final['Profile_Type'] == 'Private'])}")

# Display sample of the data
df_final.head(10)

## 11. Cleanup

In [ ]:
# Close the browser
driver.quit()
print("✅ Browser closed. Data extraction complete!")